In [ ]:
import os
import json
import numpy as np
from data.EcgQaData import EcgQaDataset
import matplotlib.pyplot as plt
from mAgents.pre_analysis_agent import pre_analysis_agent
from mAgents.data_analysis_agent import data_analysis_agent
from utils.prompt import *
from dataset.config import *
from mAgents.llm_checker import LLM_checker
from mAgents.pre_analysis_agent import pre_analysis_agent
ptbxl_dir_path = r"dataset/ecgqa_ptbxl/paraphrased/train"
EcgQaDataset_instance = EcgQaDataset(ptbxl_dir_path)
sample = {}
typelist  = []
for idx, sample in enumerate(EcgQaDataset_instance):
    qt = sample["question_type"]
    if idx == 102:
        break
question = sample["question"]

ecgs_list = sample["ecg_datas"]
        
# Make name mapping

ecgs = {}
if len(ecgs_list) == 1:
    ecgs["ecg"] = ecgs_list[0]
else:
    for index, ecg in enumerate(ecgs_list):
        ecgs[f"ecg_{index}"] = ecg

info = ecgs_list[0].base_info()
ecg_names = list(ecgs.keys())


In [ ]:
sample

In [ ]:
# Pathology Inquiry

PI_prompt = get_pathology_inquiry_prompt(question)
PI_res = pre_analysis_agent.run(PI_prompt)

In [ ]:
question_type = sample["question_type"]
DA_prompt = get_ecg_analysis_prompt(question, question_type, PI_res, ecg_names, info)
check_prompt = get_ecgqa_answer_check_prompt(question_type, sample["answer"],None)
data_analysis_agent.final_answer_checks = [LLM_checker(check_prompt).check]
data_analysis_agent.state.update(ecgs)
final_res = data_analysis_agent.run(DA_prompt)

import os
import json
import numpy as np
from data.EcgQaData import EcgQaDataset
import matplotlib.pyplot as plt
from mAgents.pre_analysis_agent import pre_analysis_agent
from mAgents.data_analysis_agent import data_analysis_agent
from utils.prompt import *
from dataset.config import ecg_qa_types
ptbxl_dir_path = r"dataset/ecgqa_ptbxl/paraphrased/train"
EcgQaDataset_instance = EcgQaDataset(ptbxl_dir_path)
sample = {}
typelist  = []
nums = 20
res  = {}
for qa_type in ecg_qa_types:
    res[qa_type] = []
for idx, sample in enumerate(EcgQaDataset_instance):
    qt = sample["question_type"]
    sample.pop("ecg_datas")
    if len(res[qt]) < nums:
        res[qt].append(sample)
    if all([len(res[qa_type]) >= nums for qa_type in ecg_qa_types]):
        break
save_path = r"dataset/ecgqa_ptbxl/sample_questions.json"
with open(save_path, "w") as f:
    json.dump(res, f, indent=4)